In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import html
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/drugscomdatasets/drug_test.csv
/kaggle/input/drugscomdatasets/drug_train.csv


In [2]:
test_data = pd.read_csv("/kaggle/input/drugscomdatasets/drug_test.csv")
train_data = pd.read_csv("/kaggle/input/drugscomdatasets/drug_train.csv")

In [3]:
# Conversion of HTML elements
train_data['review'] = train_data['review'].apply(html.unescape)
test_data['review'] = test_data['review'].apply(html.unescape)

In [4]:
train_data['rating'].value_counts()

rating
10    50745
9     27379
1     21504
8     18791
7      9395
5      7959
2      6879
3      6465
6      6301
4      4980
Name: count, dtype: int64

In [5]:
def categorize_rating(rating):
    if 1 <= rating <= 3:
        return 0  # bad
    elif 4 <= rating <= 7:
        return 1  # neutral
    elif 8 <= rating <= 10:
        return 2  # good

test_data['rating'] = test_data['rating'].apply(categorize_rating)
train_data['rating'] = train_data['rating'].apply(categorize_rating)

In [6]:
train_data['rating'].value_counts()

rating
2    96915
0    34848
1    28635
Name: count, dtype: int64

In [7]:
from sklearn.utils import resample

# En az gözlem sayısını belirle
min_count = train_data['rating'].value_counts().min()

# Dengesiz veri setini yeniden örnekle
train_data = (
    train_data
    .groupby('rating')  # Sınıf bazında gruplama
    .apply(lambda x: resample(x, replace=False, n_samples=min_count, random_state=42))  # Undersampling
    .reset_index(drop=True)
)

# Yeni veri setinin sınıf dağılımı
print(train_data['rating'].value_counts())

print(train_data.head())

rating
0    28635
1    28635
2    28635
Name: count, dtype: int64
       id       drugName            condition  \
0  143416   Etonogestrel        Birth Control   
1   93248   Vortioxetine           Depression   
2  224645  Levothyroxine  Underactive Thyroid   
3   45438     Fluoxetine   Anxiety and Stress   
4  184370    Hydroxyzine   Allergic Urticaria   

                                              review  rating  \
0  "I  don't Like being a monthly It is 7 days no...       0   
1  "This was, by far, the best med that I have ev...       0   
2  "I have had hypothyroidism for 20 years. Levox...       0   
3  "I was on this medicine for two weeks,  i felt...       0   
4  "Itching, sick to my stomach, blood constantly...       0   

                date  usefulCount  
0  February 22, 2017            0  
1   January 20, 2016            8  
2      July 14, 2009           35  
3     March 10, 2015           47  
4       July 3, 2015           15  


In [8]:
test_data['rating'].value_counts()

rating
2    32175
0    11786
1     9510
Name: count, dtype: int64

In [9]:
from sklearn.utils import resample

# En az gözlem sayısını belirle
min_count = test_data['rating'].value_counts().min()

# Dengesiz veri setini yeniden örnekle
test_data = (
    test_data
    .groupby('rating')  # Sınıf bazında gruplama
    .apply(lambda x: resample(x, replace=False, n_samples=min_count, random_state=42))  # Undersampling
    .reset_index(drop=True)
)

# Yeni veri setinin sınıf dağılımı
print(test_data['rating'].value_counts())

print(test_data.head())

rating
0    9510
1    9510
2    9510
Name: count, dtype: int64
       id         drugName       condition  \
0  154375           Reglan        Migraine   
1  217776        Strattera            ADHD   
2  144197     Etonogestrel   Birth Control   
3  155738    Metronidazole  Trichomoniasis   
4   33887  Junel Fe 1 / 20   Birth Control   

                                              review  rating  \
0  "I was given this in an IV (along with Benadry...       0   
1  "Did not seem to notice much improvement, was ...       0   
2  "I got the implant in April of this year, and ...       0   
3  "I've developed a rash on face. I had diarrhea...       0   
4  "Been on Junel for over three months now, and ...       0   

                date  usefulCount  
0  November 24, 2009         39.0  
1      June 14, 2015          6.0  
2       June 7, 2016          3.0  
3     August 2, 2012         24.0  
4  February 25, 2016          7.0  


In [10]:
# test_data.head()['review']

In [11]:
train_data['rating'].value_counts()

rating
0    28635
1    28635
2    28635
Name: count, dtype: int64

In [12]:
test_data['rating'].value_counts()

rating
0    9510
1    9510
2    9510
Name: count, dtype: int64

In [13]:
!pip install --upgrade transformers
!pip install torch
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7
  Attempting uninstall: typer
    Found existing installation: typer 0.12.5
    Uninstalling typer-0.12.5:
      Successfully uninstalled typer-0.12.5
  Attempting uninstall: huggingface-hub
    Found existing installation

In [14]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm

In [15]:
X_train = train_data['review']
X_test = test_data['review']

y_train = train_data['rating']
y_test = test_data['rating']

In [16]:
# Split the data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_data['review'].values,
    train_data['rating'].values,
    test_size=0.2,
    random_state=42
)

In [17]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        """
        Args:
            texts (List[str]): List of review texts.
            labels (List[int]): Corresponding list of labels.
            tokenizer (BertTokenizer): BERT tokenizer instance.
            max_len (int): Maximum length of tokenized sequences.
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,      # Add [CLS] and [SEP]
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',         # Pad to max_len
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',          # Return PyTorch tensors
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),         # Tensor of token IDs
            'attention_mask': encoding['attention_mask'].flatten(), # Tensor of attention masks
            'labels': torch.tensor(label, dtype=torch.long)        # Tensor of labels
        }


In [18]:
# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # Use 'bert-base-cased' if needed


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [19]:
from torch.utils.data import DataLoader

# Parameters
batch_size = 16  # Adjust based on your GPU memory

# Create datasets
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len=128)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, max_len=128)
test_dataset = SentimentDataset(X_test.values, y_test.values, tokenizer, max_len=128)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [20]:
from transformers import BertForSequenceClassification

# Load the pre-trained BERT model with a classification head
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',  # Ensure this matches the tokenizer
    num_labels=3,          # Binary classification
    output_attentions=False,
    output_hidden_states=False
)

# Move the model to GPU if available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [21]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# Initialize the optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Total number of training steps is number of batches * number of epochs
epochs = 5
total_steps = len(train_loader) * epochs

# Create the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,                # Default value in transformers examples
    num_training_steps=total_steps
)


In [22]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

best_accuracy = 0

for epoch in range(epochs):
    print(f'\n======== Epoch {epoch + 1} / {epochs} ========')
    print('Training...')
    
    model.train()
    train_loss = 0
    
    for batch in tqdm(train_loader, desc='Training'):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
    
    avg_train_loss = train_loss / len(train_loader)
    print(f'Average Training Loss: {avg_train_loss}')
    
    # Validation
    print('Validating...')
    model.eval()
    val_preds = []
    val_true = []
    val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            val_loss += loss.item()
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.cpu().numpy())
    
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = accuracy_score(val_true, val_preds)
    print(f'Validation Loss: {avg_val_loss}')
    print(f'Validation Accuracy: {val_accuracy}')
    print(classification_report(val_true, val_preds))
    
    # Save the best model
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        model.save_pretrained('best-sentiment-model')
        tokenizer.save_pretrained('best-sentiment-model')
        print("Best model saved.")



======== Epoch 1 / 5 ========
Training...


Training: 100%|██████████| 4296/4296 [25:01<00:00,  2.86it/s]


Average Training Loss: 0.6729458667983056
Validating...


Validating: 100%|██████████| 1074/1074 [02:06<00:00,  8.51it/s]

Validation Loss: 0.5857380364693743
Validation Accuracy: 0.7483266398929049
              precision    recall  f1-score   support

           0       0.79      0.78      0.79      5660
           1       0.65      0.67      0.66      5812
           2       0.81      0.80      0.80      5709

    accuracy                           0.75     17181
   macro avg       0.75      0.75      0.75     17181
weighted avg       0.75      0.75      0.75     17181



...

Average Training Loss: 0.1166378519176098
Validating...


Validating: 100%|██████████| 1074/1074 [02:05<00:00,  8.54it/s]

Validation Loss: 0.7245822714839402
Validation Accuracy: 0.8146790058785868
              precision    recall  f1-score   support

           0       0.86      0.85      0.86      5660
           1       0.73      0.78      0.76      5812
           2       0.86      0.82      0.84      5709

    accuracy                           0.81     17181
   macro avg       0.82      0.81      0.82     17181
weighted avg       0.82      0.81      0.82     17181



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved.


In [23]:
from transformers import BertForSequenceClassification

# Load the best model
best_model = BertForSequenceClassification.from_pretrained('best-sentiment-model')
best_model.to(device)
best_model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = best_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(labels.cpu().numpy())

# Display classification report
print(classification_report(test_true, test_preds))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Testing: 100%|██████████| 1784/1784 [03:29<00:00,  8.53it/s]

              precision    recall  f1-score   support

           0       0.86      0.85      0.86      9510
           1       0.73      0.78      0.75      9510
           2       0.86      0.82      0.84      9510

    accuracy                           0.82     28530
   macro avg       0.82      0.82      0.82     28530
weighted avg       0.82      0.82      0.82     28530

